# Clase 095 — K-Means: selección de K y MiniBatch

Segmentar datos no etiquetados con K-Means, elegir `K` con criterios reproducibles (elbow + silhouette) y escalar con `MiniBatchKMeans`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. K-Means sobre blobs

`make_blobs` con 5 centros; ajustamos `KMeans` con `n_init` y `random_state` fijos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

np.random.seed(42)
X, y_true = make_blobs(n_samples=2000, centers=5, cluster_std=0.5, random_state=42)

km = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)
print("inercia:", round(km.inertia_, 2), "| centroides:", km.cluster_centers_.shape)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", s=8, alpha=0.6)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c="black", marker="X", s=150, label="centroides")
plt.legend(); plt.title("K-Means (K=5) sobre make_blobs")
plt.tight_layout(); plt.show()

## 2. Elbow method: inercia vs K

La inercia siempre baja al subir `K`; buscamos el "codo" donde la mejora se aplana.

In [ ]:
Ks = range(2, 11)
inercias = []
for k in Ks:
    inercias.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_)

for k, i in zip(Ks, inercias):
    print(f"K={k}: inercia={i:.1f}")

plt.figure(figsize=(7, 4))
plt.plot(list(Ks), inercias, "o-", color="#37a")
plt.axvline(5, ls="--", color="#c33", lw=0.8, label="codo esperado (K=5)")
plt.xlabel("K"); plt.ylabel("inercia")
plt.title("Elbow method")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Silhouette: elegir K objetivamente

El `silhouette_score` (en `[-1, 1]`) combina cohesión y separación. Su máximo indica el `K` óptimo.

In [ ]:
from sklearn.metrics import silhouette_score

sils = []
for k in Ks:
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    sils.append(silhouette_score(X, labels))

k_opt = list(Ks)[int(np.argmax(sils))]
for k, s in zip(Ks, sils):
    print(f"K={k}: silhouette={s:.4f}")
print("K optimo por silhouette:", k_opt)
assert k_opt == 5, "el silhouette deberia elegir K=5"

plt.figure(figsize=(7, 4))
plt.plot(list(Ks), sils, "o-", color="#3a7")
plt.axvline(k_opt, ls="--", color="#c33", lw=0.8, label=f"K optimo = {k_opt}")
plt.xlabel("K"); plt.ylabel("silhouette score")
plt.title("Silhouette vs K")
plt.legend(); plt.tight_layout(); plt.show()

## 4. El escalado importa

Con una feature 100x mayor, K-Means la deja dominar la distancia. `StandardScaler` corrige.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

X_skew = X.copy()
X_skew[:, 1] *= 100  # una feature domina

lab_sin = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_skew)
lab_con = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(
    StandardScaler().fit_transform(X_skew))

ari_sin = adjusted_rand_score(y_true, lab_sin)
ari_con = adjusted_rand_score(y_true, lab_con)
print(f"ARI sin escalar : {ari_sin:.4f}")
print(f"ARI con escalar : {ari_con:.4f}")
assert ari_con > ari_sin, "escalar deberia mejorar el clustering"
print("StandardScaler evita que una feature domine la distancia.")

## 5. MiniBatchKMeans: velocidad vs calidad

Sobre `load_digits`, `MiniBatchKMeans` es más rápido con inercia levemente peor.

In [ ]:
import time
from sklearn.datasets import load_digits
from sklearn.cluster import MiniBatchKMeans

Xd = load_digits().data

t0 = time.perf_counter()
km_full = KMeans(n_clusters=10, n_init=10, random_state=42).fit(Xd)
t_full = time.perf_counter() - t0

t0 = time.perf_counter()
km_mb = MiniBatchKMeans(n_clusters=10, batch_size=256, n_init=10, random_state=42).fit(Xd)
t_mb = time.perf_counter() - t0

print(f"KMeans      : {t_full*1000:6.1f} ms | inercia {km_full.inertia_:.0f}")
print(f"MiniBatch   : {t_mb*1000:6.1f} ms | inercia {km_mb.inertia_:.0f}")
print("MiniBatch cambia velocidad por un poco de inercia.")

## 6. K-Means falla en `make_moons`

K-Means asume clusters convexos: parte las lunas por la mitad. Para formas arbitrarias se usa DBSCAN (clase 096).

In [ ]:
from sklearn.datasets import make_moons

Xm, ym = make_moons(n_samples=500, noise=0.05, random_state=42)
lab_m = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xm)

plt.figure(figsize=(7, 5))
plt.scatter(Xm[:, 0], Xm[:, 1], c=lab_m, cmap="coolwarm", s=12)
plt.title("K-Means parte las lunas mal (usar DBSCAN)")
plt.tight_layout(); plt.show()
print("ARI vs verdad:", round(adjusted_rand_score(ym, lab_m), 4), "(bajo: K-Means no sirve aca)")

## Ejercicios

1. Ajustá `KMeans(n_clusters=5)` sobre blobs y graficá puntos + `cluster_centers_`.
2. Calculá `inertia_` y `silhouette_score` para `K = 2..10` y justificá el `K` elegido.
3. Repetí el ajuste sin escalar una feature 100x mayor y comparalo con `StandardScaler` previo.
4. Sobre `load_digits`, cronometrá `KMeans` vs `MiniBatchKMeans` y compará inercias.
5. Corré K-Means sobre `make_moons` y explicá visualmente por qué falla.

## Conclusiones

- **Escalá siempre** antes de K-Means: usa distancia euclídea y una feature grande domina.
- Fijá `n_init >= 10` y `random_state` para reproducibilidad (K-Means depende de la inicialización).
- Elegí `K` combinando elbow (inercia) + silhouette; priorizá silhouette si no hay codo claro.
- `MiniBatchKMeans` acelera a costa de un poco de inercia; K-Means no sirve para clusters no convexos.